# DaTSCAN: protocolos latentes y validación agrupada

Este notebook integra las cuatro etapas: inventario y extracción de metadatos, clustering de protocolos, creación de folds agrupados y auditoría. Ejecute las celdas en orden.

**Regla metodológica:** los clústeres se construyen únicamente con variables de adquisición y geometría. La etiqueta diagnóstica se usa después, solo para auditar y equilibrar los folds.

## 0. Instalación (solo la primera vez)

In [ ]:
# Descomente y ejecute si faltan paquetes:
# %pip install numpy pandas nibabel scikit-learn scipy matplotlib seaborn joblib

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import os
import hashlib, json, warnings
import joblib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# RUTAS REALES DEL PROYECTO EN SU COMPUTADORA.
DATA_ROOT = Path(r'C:\Users\DELL\OneDrive\Escritorio\kaggle\parkinson')
NIFTI_DIR = DATA_ROOT / 'niftis_utCGpHE'
LABELS_CSV = DATA_ROOT / 'train_labels_JNDlMjr.csv'

UID_COLUMN = None       # Ejemplo: 'StudyInstanceUID'; None = detectar automáticamente
TARGET_COLUMN = None    # Ejemplo: 'target'; None = detectar automáticamente

PROJECT_DIR = DATA_ROOT / 'latent_protocol_cv'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
FIGURE_DIR = PROJECT_DIR / 'figures'
for folder in (OUTPUT_DIR, ARTIFACT_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

SAMPLE_VOXELS = 200_000
FOREGROUND_RELATIVE_THRESHOLD = 0.01
K_MIN, K_MAX = 3, 10
SELECTED_K = None       # Déjelo en None para selección automática
MIN_CLUSTER_SIZE = 80
PCA_VARIANCE = 0.95
N_INIT = 50
RANDOM_STATE = 20260910
STABILITY_REPEATS = 10
N_SPLITS = 5

print('NIfTI:', NIFTI_DIR.resolve(), '| existe:', NIFTI_DIR.exists())
print('Etiquetas:', LABELS_CSV.resolve(), '| existe:', LABELS_CSV.exists())
if not NIFTI_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta: {NIFTI_DIR}')
if not LABELS_CSV.exists():
    raise FileNotFoundError(f'No existe el CSV: {LABELS_CSV}')

## 2. Funciones auxiliares
Esta celda detecta las columnas, empareja exactamente cada UID y extrae información del encabezado NIfTI.

In [ ]:
UID_CANDIDATES = ('uid','study_uid','studyinstanceuid','StudyInstanceUID','patient_id','scan_id','image_id','id')
TARGET_CANDIDATES = ('target','label','y','abnormal','is_abnormal','pathological','is_pathological','diagnosis')

def detect_column(df, configured, candidates, role, required=True):
    if configured:
        if configured not in df.columns:
            raise KeyError(f'La columna {role} configurada no existe: {configured}')
        return configured
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise KeyError(f'No se detectó la columna {role}. Disponibles: {list(df.columns)}')
    return None

def normalize_uid(value):
    value = str(value).strip()
    for suffix in ('.nii.gz', '.nii'):
        if value.endswith(suffix):
            value = value[:-len(suffix)]
    return value

def deterministic_sample(values, n, uid):
    flat = values.reshape(-1)
    if flat.size <= n:
        return flat
    seed = int(hashlib.sha256(uid.encode('utf-8')).hexdigest()[:8], 16)
    return flat[np.random.default_rng(seed).choice(flat.size, size=n, replace=False)]

def safe_float(value):
    try:
        return float(np.asarray(value).reshape(-1)[0])
    except (TypeError, ValueError, IndexError):
        return np.nan

def extract_one(path, uid):
    img = nib.load(path, mmap=True)
    header = img.header
    shape = tuple(int(v) for v in img.shape)
    if len(shape) < 3:
        raise ValueError(f'Volumen con menos de tres dimensiones: {path} {shape}')
    zooms = tuple(float(v) for v in header.get_zooms()[:3])
    affine = np.asarray(img.affine, dtype=float)
    data = np.asarray(img.dataobj, dtype=np.float32)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    sampled = deterministic_sample(data, SAMPLE_VOXELS, uid)
    nonzero = sampled[sampled > 0]
    basis = nonzero if nonzero.size >= 100 else sampled
    p = np.percentile(basis, [1,5,25,50,75,90,95,99,99.5])
    threshold = max(0.0, FOREGROUND_RELATIVE_THRESHOLD * float(p[-1]))
    spatial_shape = shape[:3]
    fov = tuple(spatial_shape[i] * zooms[i] for i in range(3))
    return {
        'uid': uid, 'nifti_path': str(path.resolve()),
        'shape_x': spatial_shape[0], 'shape_y': spatial_shape[1], 'shape_z': spatial_shape[2],
        'ndim': len(shape), 'n_voxels': int(np.prod(spatial_shape)),
        'spacing_x': zooms[0], 'spacing_y': zooms[1], 'spacing_z': zooms[2],
        'voxel_volume_mm3': float(np.prod(zooms)),
        'fov_x_mm': fov[0], 'fov_y_mm': fov[1], 'fov_z_mm': fov[2],
        'orientation': ''.join(nib.aff2axcodes(affine)),
        'affine_det_sign': int(np.sign(np.linalg.det(affine[:3,:3]))),
        'qform_code': int(header['qform_code']), 'sform_code': int(header['sform_code']),
        'dtype': str(header.get_data_dtype()),
        'scl_slope': safe_float(header.get('scl_slope', np.nan)),
        'scl_inter': safe_float(header.get('scl_inter', np.nan)),
        'intensity_min': float(sampled.min()), 'intensity_max': float(sampled.max()),
        'intensity_mean': float(sampled.mean()), 'intensity_std': float(sampled.std()),
        **{f'intensity_{name}': float(v) for name, v in zip(('p01','p05','p25','p50','p75','p90','p95','p99','p995'), p)},
        'nonzero_fraction': float(np.mean(sampled > 0)),
        'foreground_fraction': float(np.mean(sampled > threshold))
    }

## 3. Inventario y correspondencia entre imágenes y etiquetas

In [ ]:
files = sorted(NIFTI_DIR.rglob('*.nii.gz')) + sorted(NIFTI_DIR.rglob('*.nii'))
if not files:
    raise FileNotFoundError(f'No se encontraron NIfTI en {NIFTI_DIR.resolve()}')
all_images = pd.DataFrame({'uid': [normalize_uid(p.name) for p in files], 'nifti_path': [str(p.resolve()) for p in files]})
if all_images['uid'].duplicated().any():
    raise ValueError('Hay UID duplicados entre los archivos NIfTI.')

labels_raw = pd.read_csv(LABELS_CSV)
uid_col = detect_column(labels_raw, UID_COLUMN, UID_CANDIDATES, 'UID')
target_col = detect_column(labels_raw, TARGET_COLUMN, TARGET_CANDIDATES, 'objetivo')
labels = labels_raw[[uid_col, target_col]].copy()
labels.columns = ['uid', 'target']
labels['uid'] = labels['uid'].map(normalize_uid)
if labels['uid'].duplicated().any():
    raise ValueError('El CSV contiene UID duplicados.')

matching = all_images[['uid']].merge(labels[['uid']], on='uid', how='outer', indicator=True)
display(matching['_merge'].value_counts().rename_axis('correspondencia').to_frame('n'))
missing_images = matching[matching['_merge'] == 'right_only']
if len(missing_images):
    display(missing_images.head(20))
    raise ValueError(f'Faltan {len(missing_images)} imágenes del conjunto etiquetado.')

# Para construir la CV se conservan únicamente los estudios con etiqueta.
images = all_images.merge(labels[['uid']], on='uid', how='inner', validate='one_to_one')
unlabeled_images = all_images[~all_images['uid'].isin(labels['uid'])].copy()
print(f'NIfTI totales: {len(all_images):,}')
print(f'NIfTI de entrenamiento: {len(images):,}')
print(f'NIfTI sin etiqueta: {len(unlabeled_images):,}')
print(f'Etiquetas: {len(labels):,}')
display(images.head())

## 4. Extracción de metadatos
Esta es la etapa más lenta porque abre todos los estudios. El archivo se guarda como punto de control.

In [ ]:
metadata_path = OUTPUT_DIR / 'protocol_metadata.csv'
RECALCULATE_METADATA = False

if metadata_path.exists() and not RECALCULATE_METADATA:
    metadata = pd.read_csv(metadata_path)
    print('Se reutilizó:', metadata_path.resolve())
else:
    rows = []
    for i, row in images.iterrows():
        rows.append(extract_one(Path(row['nifti_path']), row['uid']))
        if (i + 1) % 50 == 0 or i + 1 == len(images):
            print(f'Procesados {i + 1}/{len(images)}')
    metadata = pd.DataFrame(rows).merge(labels, on='uid', how='left', validate='one_to_one')
    metadata.to_csv(metadata_path, index=False)
    print('Guardado:', metadata_path.resolve())

print('Dimensiones:', metadata.shape)
display(metadata.head())
display(metadata[['shape_x','shape_y','shape_z','spacing_x','spacing_y','spacing_z','orientation','dtype']].describe(include='all').T)

## 5. Selección de variables de protocolo
Las intensidades absolutas y la etiqueta se excluyen del clustering porque podrían contener señal diagnóstica.

In [ ]:
EXCLUDED = {
    'uid','nifti_path','target','cluster','fold',
    'intensity_min','intensity_max','intensity_mean','intensity_std',
    'intensity_p01','intensity_p05','intensity_p25','intensity_p50',
    'intensity_p75','intensity_p90','intensity_p95','intensity_p99','intensity_p995'
}
numeric_features = [c for c in metadata.select_dtypes(include=np.number).columns if c not in EXCLUDED]
categorical_features = [c for c in ('orientation','dtype') if c in metadata.columns]
print('Numéricas:', numeric_features)
print('Categóricas:', categorical_features)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scaler', RobustScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([('numeric', numeric_pipe, numeric_features), ('categorical', categorical_pipe, categorical_features)], remainder='drop')
x_pre = preprocessor.fit_transform(metadata)
pca = PCA(n_components=PCA_VARIANCE, svd_solver='full')
x_protocol = pca.fit_transform(x_pre)
print(f'Matriz preprocesada: {x_pre.shape}; componentes PCA: {x_protocol.shape[1]}; varianza: {pca.explained_variance_ratio_.sum():.3f}')

## 6. Comparación del número de clústeres

In [ ]:
def clustering_stability(x, k):
    reference = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT).fit_predict(x)
    scores = []
    for j in range(1, STABILITY_REPEATS + 1):
        candidate = KMeans(n_clusters=k, random_state=RANDOM_STATE+j, n_init=N_INIT).fit_predict(x)
        scores.append(adjusted_rand_score(reference, candidate))
    return float(np.mean(scores))

candidates = []
for k in range(K_MIN, K_MAX + 1):
    labels_k = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT).fit_predict(x_protocol)
    sizes = pd.Series(labels_k).value_counts()
    candidates.append({'k': k, 'silhouette': silhouette_score(x_protocol, labels_k), 'stability_ari': clustering_stability(x_protocol, k), 'min_cluster_size': int(sizes.min()), 'max_cluster_size': int(sizes.max()), 'size_ratio': sizes.max()/sizes.min()})
candidate_df = pd.DataFrame(candidates)
candidate_df.to_csv(OUTPUT_DIR / 'protocol_cluster_candidates.csv', index=False)
display(candidate_df.style.format({'silhouette':'{:.3f}','stability_ari':'{:.3f}','size_ratio':'{:.2f}'})
        .background_gradient(subset=['silhouette','stability_ari'], cmap='Greens'))

eligible = candidate_df[candidate_df['min_cluster_size'] >= MIN_CLUSTER_SIZE]
if SELECTED_K is None:
    if eligible.empty:
        raise ValueError('Ningún k cumple el tamaño mínimo. Revise la tabla antes de continuar.')
    selected_k = int(eligible.sort_values(['silhouette','stability_ari'], ascending=False).iloc[0]['k'])
else:
    selected_k = int(SELECTED_K)
print('K seleccionado:', selected_k)

## 7. Modelo final de clustering e interpretación descriptiva

In [ ]:
clusterer = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=N_INIT)
metadata_clustered = metadata.copy()
metadata_clustered['protocol_cluster'] = clusterer.fit_predict(x_protocol)
metadata_clustered.to_csv(OUTPUT_DIR / 'protocol_metadata_clustered.csv', index=False)
joblib.dump({'preprocessor':preprocessor,'pca':pca,'clusterer':clusterer,'numeric_features':numeric_features,'categorical_features':categorical_features}, ARTIFACT_DIR / 'protocol_cluster_pipeline.joblib')

cluster_summary = metadata_clustered.groupby('protocol_cluster').agg(
    n=('uid','size'), normal=('target',lambda z:int((z==0).sum())), pathological=('target',lambda z:int((z==1).sum())), prevalence=('target','mean'),
    shape_x=('shape_x','median'), shape_y=('shape_y','median'), shape_z=('shape_z','median'),
    spacing_x=('spacing_x','median'), spacing_y=('spacing_y','median'), spacing_z=('spacing_z','median'),
    fov_x_mm=('fov_x_mm','median'), fov_y_mm=('fov_y_mm','median'), fov_z_mm=('fov_z_mm','median')
).reset_index()
cluster_summary.to_csv(OUTPUT_DIR / 'protocol_cluster_summary.csv', index=False)
display(cluster_summary.style.format({'prevalence':'{:.3f}'}))

plot_df = pd.DataFrame({'PC1':x_protocol[:,0], 'PC2':x_protocol[:,1] if x_protocol.shape[1]>1 else 0, 'cluster':metadata_clustered['protocol_cluster'].astype(str)})
plt.figure(figsize=(9,7))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='cluster', s=38, alpha=.75)
plt.title('Protocolos latentes en el espacio PCA')
plt.tight_layout(); plt.savefig(FIGURE_DIR/'protocol_pca_clusters.png', dpi=180); plt.show()

## 8. Construcción de folds agrupados
Si el número de clústeres coincide con el número de folds, se aplica *leave-one-cluster-out*. Si hay más clústeres, se usa `StratifiedGroupKFold`. Ningún clúster puede repartirse entre folds.

In [ ]:
groups = metadata_clustered['protocol_cluster'].to_numpy()
unique_groups = np.unique(groups)
folds = metadata_clustered[['uid','target','protocol_cluster']].copy()
folds['fold'] = -1

if len(unique_groups) == N_SPLITS:
    folds['fold'] = folds['protocol_cluster'].map({g:i for i,g in enumerate(sorted(unique_groups))}).astype(int)
    fold_method = 'leave_one_cluster_out'
elif len(unique_groups) > N_SPLITS:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (_, valid_idx) in enumerate(cv.split(np.zeros(len(folds)), folds['target'], groups=groups)):
        folds.loc[valid_idx, 'fold'] = fold
    fold_method = 'stratified_group_kfold'
else:
    raise ValueError(f'Hay {len(unique_groups)} clústeres, menos que los {N_SPLITS} folds solicitados.')

if (folds.groupby('protocol_cluster')['fold'].nunique() > 1).any():
    raise RuntimeError('Fuga: un clúster fue dividido entre folds.')
folds.to_csv(OUTPUT_DIR/'train_protocol_folds.csv', index=False)
fold_summary = folds.groupby('fold').agg(n=('uid','size'), normal=('target',lambda z:int((z==0).sum())), pathological=('target',lambda z:int((z==1).sum())), prevalence=('target','mean'), n_clusters=('protocol_cluster','nunique')).reset_index()
fold_summary.to_csv(OUTPUT_DIR/'fold_summary.csv', index=False)
print('Método:', fold_method)
display(fold_summary.style.format({'prevalence':'{:.3f}'}))

## 9. Auditoría de clústeres y folds

In [ ]:
contingency = pd.crosstab(metadata_clustered['protocol_cluster'], metadata_clustered['target'])
chi2, p_value, _, _ = chi2_contingency(contingency, correction=False)
n_total = contingency.to_numpy().sum()
r, c = contingency.shape
cramers_v = float(np.sqrt((chi2/n_total) / max(1, min(c-1, r-1))))
cluster_sizes = metadata_clustered['protocol_cluster'].value_counts()
min_normal = min(int((g['target']==0).sum()) for _,g in folds.groupby('fold'))
min_pathological = min(int((g['target']==1).sum()) for _,g in folds.groupby('fold'))
report = {
    'n':len(folds), 'n_clusters':int(folds['protocol_cluster'].nunique()), 'n_folds':int(folds['fold'].nunique()),
    'cluster_size_min':int(cluster_sizes.min()), 'cluster_size_max':int(cluster_sizes.max()),
    'cluster_target_chi2':float(chi2), 'cluster_target_p_value':float(p_value), 'cluster_target_cramers_v':cramers_v,
    'min_normal_per_fold':min_normal, 'min_pathological_per_fold':min_pathological,
    'global_prevalence':float(folds['target'].mean()), 'fold_prevalence_min':float(fold_summary['prevalence'].min()), 'fold_prevalence_max':float(fold_summary['prevalence'].max()),
    'cluster_split_across_folds':bool((folds.groupby('protocol_cluster')['fold'].nunique()>1).any())
}
report['status'] = 'review' if cramers_v >= .30 or min(min_normal,min_pathological) < 20 else 'provisional_accept'
with open(OUTPUT_DIR/'audit_report.json','w',encoding='utf-8') as f:
    json.dump(report,f,ensure_ascii=False,indent=2)
display(pd.Series(report, name='resultado').to_frame())
display(contingency.style.background_gradient(cmap='Blues'))

prevalence_plot = metadata_clustered.groupby('protocol_cluster')['target'].agg(n='size',prevalence='mean').reset_index()
plt.figure(figsize=(9,6))
sns.barplot(data=prevalence_plot,x='protocol_cluster',y='prevalence',color='#2D6A4F')
plt.axhline(folds['target'].mean(),color='#B44343',linestyle='--',label='Prevalencia global')
plt.ylim(0,1); plt.xlabel('Clúster de protocolo'); plt.ylabel('Proporción patológica'); plt.legend()
plt.tight_layout(); plt.savefig(FIGURE_DIR/'cluster_prevalence.png',dpi=180); plt.show()

## 10. Decisión antes de entrenar

No continúe automáticamente al modelado. Revise:

- que los clústeres se expliquen por dimensiones, espaciado, orientación o campo de visión;
- que no haya clústeres demasiado pequeños;
- que `cluster_target_cramers_v` no sea alto;
- que cada fold contenga suficientes casos normales y patológicos;
- que `cluster_split_across_folds` sea `False`.

Comparta las tablas y los dos gráficos obtenidos antes de iniciar ROI, Fourier, wavelets o redes neuronales.